# Pronóstico mensual del IPAP BMC

El modelo ETS usa hasta 120 meses recientes y pronostica doce meses completos. El CSV conserva 189 observaciones oficiales. Seguiremos los datos, el entrenamiento, la evaluación y el modelo guardado. Los períodos ya se examinaron durante el desarrollo, de modo que la evaluación es retrospectiva.


In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from modelo_ipap import (BASE, VENTANA, cargar_datos, pronosticar, evaluar, metricas,
                         guardar_modelo, cargar_modelo, pronostico_guardado, graficar)
plt.rcParams.update({"font.family":"Arial", "font.size":10})


## Leer los datos

Cada fila representa un mes. fecha indica el período y valor contiene puntos del Total agropecuario, no pesos. Se comprueba la huella del CSV, la continuidad, los duplicados y los valores inválidos. No se rellenan meses.


In [ ]:
serie = cargar_datos()
print(f"{len(serie)} meses: {serie.index[0]} a {serie.index[-1]}")
display(serie.head(3).to_frame())
display(serie.tail(3).to_frame())
fig, eje = plt.subplots(figsize=(9,3))
eje.plot(serie.index.to_timestamp(), serie.to_numpy(), color="black")
eje.set(xlabel="Mes", ylabel="Puntos del IPAP")
fig.tight_layout()
plt.show()


## Separar el entrenamiento y la evaluación

Para pronosticar 2025 nos situamos en diciembre de 2024. El modelo solo recibe información hasta ese corte y utiliza los últimos 120 meses disponibles. Los doce valores reales de 2025 se guardan aparte para medir el error después.


In [ ]:
origen = "2024-12"
historia = serie.loc[:origen]
entrenamiento = historia.iloc[-VENTANA:]
reales_2025 = serie.loc["2025-01":"2025-12"]
print(f"Entrenamiento: {len(entrenamiento)} meses, {entrenamiento.index[0]} a {entrenamiento.index[-1]}")
print(f"Evaluación: {len(reales_2025)} meses de 2025")


## Ajustar ETS y pronosticar

ETS combina nivel, tendencia aditiva amortiguada y estacionalidad aditiva de doce meses. El ajuste estima sus parámetros dentro del entrenamiento. pronosticar aplica el límite de 120 meses y devuelve los doce meses siguientes sin actualizar con los datos reales intermedios. Se utiliza ExponentialSmoothing de statsmodels 0.14.6.


In [ ]:
prediccion = pronosticar(historia)
comparacion = pd.DataFrame({"real_BMC":reales_2025,"prediccion_ETS":prediccion})
comparacion["error_real_menos_predicho"] = comparacion.real_BMC-comparacion.prediccion_ETS
display(comparacion.round(2))


## Comparar con reglas sencillas

Naive repite el último valor. Naive12 repite el mismo mes del año anterior. Deriva12 prolonga el cambio mensual medio del último año. Todos pronostican las mismas fechas. RMSE y MAE se expresan en puntos; MAPE en porcentaje. Un número menor indica menor error.


In [ ]:
evaluacion_2025 = evaluar(serie,[origen])
display(metricas(evaluacion_2025).round(4))


## Evaluar seis cortes

Usamos cortes de septiembre de 2020 a septiembre de 2025 y siempre generamos los doce meses posteriores. Son 72 objetivos por método. Los bloques evaluados no se superponen, pero los entrenamientos comparten historia. La comparación de 2025 comparte meses con estos bloques y no es otra prueba independiente.


In [ ]:
bloques = evaluar(serie)
display(metricas(bloques).round(4))
por_corte = metricas(bloques,("origen","modelo"))
display(por_corte.pivot(index="origen",columns="modelo",values="RMSE").round(2))


## Leer el error por horizonte

El horizonte 1 es el primer mes después del corte y el horizonte 12 es el último de ese año. Cada horizonte reúne seis errores por método. La precisión del primer mes no representa la precisión del año completo.


In [ ]:
por_horizonte = metricas(bloques,("modelo","horizonte"))
display(por_horizonte[por_horizonte.modelo=="ETS_120_meses"].round(2))


## Límites de la comparación

La captura puede incorporar revisiones. Sin versiones históricas no podemos reconstruir toda la información disponible en cada fecha. La ventana se definió durante el desarrollo sobre períodos ya examinados. Por eso la evaluación es retrospectiva y no garantiza precisión futura. La entrega contiene pronósticos centrales, sin intervalos calibrados.


## Guardar el ajuste y ver los gráficos

El modelo final utiliza 120 meses, de octubre de 2016 a septiembre de 2026. Sus predicciones cubren octubre de 2026 a septiembre de 2027 y no se incorporan al CSV oficial. El archivo PKL guarda el ajuste y permite usarlo sin entrenar otra vez. El programa principal genera las mismas tablas y figuras de este recorrido.


In [ ]:
guardar_modelo(serie)
paquete = cargar_modelo()
futuro = pronostico_guardado(paquete)
futuro.to_csv(BASE/"resultados/pronostico_12_meses.csv",index=False)
graficar(serie,bloques,evaluacion_2025,futuro)
print(f"Modelo entrenado con {len(paquete["historia"])} meses")
display(futuro[["fecha","horizonte","prediccion"]].round(2))
for nombre in ["comparacion_2025.png","error_por_corte.png","error_por_horizonte.png","pronostico_12_meses.png"]:
    display(Image(filename=str(BASE/"resultados"/nombre)))
